# 04 Results Summary

Resum dels candidats a grups de comfort a partir dels 7 grups originals 

Tenim 4 principals branques

1. **ML recoverability** -> es poden predir bé
2. **Classical separability** — estan ben separats estadísticament?
3. **Thermal-regime sharpness** — estan ben separats across T?
4. **Balance / practical robustness** — estan ben balancejats??

La idea és agafar la que millors mètriques globals pugui tenir


## Candidate outcome definitions used in this notebook

### 3-group candidates
- **comfort3**  
  `VC + C + SC | N | SU + U + VU`

- **comfort3_option1**  
  `VC + C + SC + N | SU + U | VU`

- **comfort3_option2**  
  `VC + C + SC | N + SU + U | VU`

- **comfort3_UNoption**  
  `VC + C + SC + N + SU | U | VU`

### 4-group candidates
- **comfort4_soft**  
  `VC + C | SC + N | SU | U + VU`

- **comfort4_option1**  
  `VC | C + SC + N | SU + U | VU`

- **comfort4_option2**  
  `VC + C + SC | N | SU | U + VU`

These are linked to the exact contiguous partitions stored in `comfort_partition_search.csv`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Optional: nicer display in notebook
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 200)


def find_csv(name: str) -> Path:
    """Find a CSV whether the notebook is run from the project folder or directly from uploaded files."""
    candidates = [
        Path("csvs") / name,
        Path(".") / name,
        Path("..") / "csvs" / name,
        Path("..") / name,
        Path("/mnt/data") / name,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {name} in any expected location.")


CSV_DIR = Path("csvs")
FIG_DIR = Path("figures")
CSV_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)


In [ ]:
# ----------------------------
# Load input CSVs
# ----------------------------
ml_raw = pd.read_csv(find_csv("comfort_partition_search.csv"))
kw = pd.read_csv(find_csv("new_cat_kruskal_wallis_results.csv"))
sp = pd.read_csv(find_csv("new_cat_sp_df.csv"))
pw = pd.read_csv(find_csv("new_cat_pairwise_adjacent_mannwhitney_results.csv"))
chi = pd.read_csv(find_csv("new_cat_chi_square_cramersv_results.csv"))
temp = pd.read_csv(find_csv("ALL_tempbin_scorecard.csv"))

print("Loaded:")
print("  ML rows:", len(ml_raw))
print("  Kruskal rows:", len(kw))
print("  Spearman rows:", len(sp))
print("  Pairwise rows:", len(pw))
print("  Chi-square rows:", len(chi))
print("  Temp-bin rows:", len(temp))


In [ ]:
# ----------------------------
# Map ML partitions to your named outcomes
# ----------------------------
partition_to_outcome = {
    "Very comfortable + Comfortable + Slightly comfortable || Neutral || Slightly uncomfortable + Uncomfortable + Very uncomfortable": "comfort3",
    "Very comfortable + Comfortable + Slightly comfortable + Neutral || Slightly uncomfortable + Uncomfortable || Very uncomfortable": "comfort3_option1",
    "Very comfortable + Comfortable + Slightly comfortable || Neutral + Slightly uncomfortable + Uncomfortable || Very uncomfortable": "comfort3_option2",
    "Very comfortable + Comfortable || Slightly comfortable + Neutral || Slightly uncomfortable || Uncomfortable + Very uncomfortable": "comfort4_soft",
    "Very comfortable || Comfortable + Slightly comfortable + Neutral || Slightly uncomfortable + Uncomfortable || Very uncomfortable": "comfort4_option1",
    "Very comfortable + Comfortable + Slightly comfortable || Neutral || Slightly uncomfortable || Uncomfortable + Very uncomfortable": "comfort4_option2",
    "Very comfortable + Comfortable + Slightly comfortable + Neutral + Slightly uncomfortable || Uncomfortable || Very uncomfortable": "comfort3_UNoption",
}

pretty_labels = {
    "comfort3": "Comfort 3",
    "comfort3_option1": "Comfort 3 – option 1",
    "comfort3_option2": "Comfort 3 – option 2",
    "comfort3_UNoption": "Comfort 3 – UN option",
    "comfort4_soft": "Comfort 4 soft",
    "comfort4_option1": "Comfort 4 – option 1",
    "comfort4_option2": "Comfort 4 – option 2",
}

main_outcomes = [
    "comfort3",
    "comfort3_option1",
    "comfort3_option2",
    "comfort4_soft",
    "comfort4_option1",
]

mapping_table = pd.DataFrame(
    [{"outcome": v, "partition": k} for k, v in partition_to_outcome.items()]
).sort_values("outcome").reset_index(drop=True)

mapping_table


## Aggregate the evidence blocks

For each candidate outcome, this section computes:

- **ML summary** from the best row in `comfort_partition_search.csv`
- **Spearman summary** from `new_cat_sp_df.csv`
- **Adjacent-pair MWU summary** from `new_cat_pairwise_adjacent_mannwhitney_results.csv`
- **Kruskal summary** from `new_cat_kruskal_wallis_results.csv`
- **Chi-square / Cramér's V summary** from `new_cat_chi_square_cramersv_results.csv`
- **Temperature-bin summary** from `ALL_tempbin_scorecard.csv`


## Què representa el dataframe `results`

El dataframe `results` és la taula central de síntesi del notebook de resultats.  
Cada fila correspon a una **coarse-graining candidata** de l’escala original de 7 nivells de thermal comfort.

Per tant, `results` no és un output d’un sol test, sinó una **fusió** de diversos blocs d’evidència:

1. **Machine learning**
2. **Estadística clàssica**
3. **Estructura per bins de temperatura**
4. **Balanceig de classes**
5. **Scores sintètics finals**

L’objectiu és comparar diferents agrupacions de categories com a **espais d’estat efectius** del confort tèrmic.

---

## 1. Columnes d’identificació

### `outcome`
Nom intern de l’agrupació candidata.

Exemples:
- `comfort3`
- `comfort3_option1`
- `comfort3_option2`
- `comfort4_soft`
- `comfort4_option1`

### `pretty_label`
Nom més llegible per mostrar en taules i figures.

---

## 2. Columnes del millor model de ML

Aquestes columnes venen del fitxer de partition search de ML.  
Per a cada outcome, es selecciona **la millor fila de ML** segons:

1. `bal_acc_adj_chance`
2. `f1_macro_mean`
3. `bal_acc_penalized`

### `feature_set`
Conjunt de predictors utilitzat en la millor fila de ML per aquell outcome.

Exemples:
- només temperatura
- només HDX
- temperatura + context
- models amb `thermal_comfort_walking`

Això indica si l’agrupació és recuperable només amb variables físiques/contextuals o si necessita informació subjectiva addicional.

### `model`
Família de model associada a la millor fila.

Valors típics:
- `rf` = random forest
- `logreg` = logistic regression

---

## 3. Mètriques principals de ML

### `bal_acc_mean`
Balanced accuracy mitjana.

És la mitjana del recall per classe, així que és més justa que l’accuracy normal quan les classes estan desbalancejades.

Interpretació:
- més alt = millor recuperabilitat global de les classes

### `f1_macro_mean`
Macro-F1 mitjà.

Fa la mitjana de l’F1 de cada classe, donant el mateix pes a totes.

Interpretació:
- més alt = millor compromís precisió/recall entre classes

### `bal_acc_adj_chance`
Balanced accuracy ajustada respecte a l’atzar.

Escala:
- `0` = nivell atzar
- `1` = classificació perfecta

És útil perquè permet comparar problemes amb diferent nombre de classes.

### `bal_acc_penalized`
Versió penalitzada de la balanced accuracy.

Redueix la puntuació quan una agrupació té una classe massa dominant.

Interpretació:
- més alt = bona recuperabilitat **i** menys risc que el resultat estigui “inflant-se” per una classe molt gran

---

## 4. Mètriques de balanceig de classes

### `largest_class_share`
Proporció de mostres que pertanyen a la classe més gran.

Interpretació:
- alt = una classe domina massa
- massa alt = l’agrupació pot ser massa col·lapsada

### `smallest_class_share`
Proporció de mostres de la classe més petita.

Interpretació:
- molt petit = classe rara, potencialment inestable

### `entropy_norm`
Entropia normalitzada de la distribució de classes.

Interpretació:
- prop de 1 = agrupació més equilibrada
- més baix = distribució més dominada per una o poques classes

---

## 5. Mètriques de monotonicitat (Spearman)

Aquestes columnes venen del CSV `new_cat_sp_df.csv`.

Mesuren fins a quin punt els predictors tenen una relació **monòtona** amb l’outcome ordenat.

### `n_sig_rho`
Nombre de predictors amb Spearman significatiu (`p < 0.05`).

Interpretació:
- més alt = més variables segueixen una tendència ordinal clara amb l’outcome

### `median_abs_rho`
Mediana de `|rho|` entre tots els predictors.

Interpretació:
- més alt = millor estructura ordinal global

### `max_abs_rho`
Valor màxim de `|rho|`.

Interpretació:
- predictor individual amb relació monòtona més forta

---

## 6. Mètriques de separació adjacent

Aquestes columnes venen del CSV `new_cat_pairwise_adjacent_mannwhitney_results.csv`.

Mesuren si les **classes veïnes** dins l’agrupació són realment separables.

### `n_adj_pairs`
Nombre total de comparacions adjacents disponibles.

### `n_sig_adj`
Nombre de comparacions adjacents significatives després de correcció de Holm.

Interpretació:
- més alt = millor separació entre estats consecutius

### `median_abs_rb`
Mediana de `|rank-biserial correlation|` per a les comparacions adjacents.

Interpretació:
- més alt = separació adjacent típica més forta

### `max_abs_rb`
Valor màxim de `|rank-biserial|`.

Interpretació:
- separació adjacent més forta observada

---

## 7. Mètriques de separació global numèrica (Kruskal–Wallis)

Aquestes columnes venen del CSV `new_cat_kruskal_wallis_results.csv`.

### `n_sig_kw`
Nombre de predictors numèrics amb resultat significatiu (`p < 0.05`).

Interpretació:
- més alt = més variables numèriques separen les classes

### `median_H`
Mediana de l’estadístic `H` de Kruskal.

Interpretació:
- més alt = separació global més forta en l’espai de predictors numèrics

### `max_H`
Valor màxim de `H`.

Interpretació:
- predictor numèric amb separació més forta

---

## 8. Mètriques d’associació categòrica (Chi-square / Cramér’s V)

Aquestes columnes venen del CSV `new_cat_chi_square_cramersv_results.csv`.

### `n_sig_chi`
Nombre de predictors categòrics amb associació significativa.

### `median_v`
Mediana de Cramér’s V.

Interpretació:
- més alt = associació típica més forta amb variables categòriques/contextuals

### `max_v`
Valor màxim de Cramér’s V.

Interpretació:
- associació categòrica individual més forta

---

## 9. Mètriques d’estructura per bins de temperatura

Aquestes columnes venen del resum de `ALL_tempbin_scorecard.csv`.

Mesuren fins a quin punt l’agrupació mostra una **estructura de règim tèrmic** clara quan es divideixen les dades en bins de temperatura.

### `tempbin_score`
Score sintètic de qualitat per estructura tèrmica.

Interpretació:
- més alt = agrupació més coherent amb la separació per règims de temperatura

### `mean_abs_prop_diff`
Diferència absoluta mitjana entre proporcions de classe dins dels bins.

Interpretació:
- més alt = més separació entre classes dins dels bins

### `mean_ci_overlap`
Solapament mitjà absolut dels intervals de confiança entre classes dins dels bins.

### `mean_ci_overlap_norm`
Versió normalitzada del solapament dels intervals de confiança.

Interpretació:
- més baix = les classes es distingeixen millor dins dels bins

### `max_abs_prop_diff`
Diferència màxima observada de proporcions entre classes

### `min_ci_overlap`
Solapament mínim observat d’intervals de confiança

---

## 10. Mètriques de dominància i entropia per bins

### `mean_dominant_prop`
Proporció mitjana de la classe dominant a cada bin.

Interpretació:
- més alt = cada bin tendeix a estar més dominat per una sola classe

### `max_dominant_prop`
Màxima dominància observada en algun bin

### `mean_entropy_norm`
Entropia normalitzada mitjana dels bins

Interpretació:
- més baixa = bins menys barrejats, estructura de règim més forta

### `min_entropy_norm`
Entropia mínima observada en algun bin

### `mean_concentration`
Índex mitjà de concentració (tipus suma de \(p_i^2\))

Interpretació:
- més alt = bins més concentrats en poques classes

### `mean_l1_imbalance`
Distància mitjana respecte a una distribució uniforme dins els bins

Interpretació:
- més alt = més desbalanceig dins dels bins

---

## 11. Scores compostos

Aquests scores no són variables observades directament, sinó resums normalitzats per comparar millor les agrupacions.

### `ml_score`
Score sintètic de recuperabilitat per ML.

Combina:
- `bal_acc_adj_chance`
- `f1_macro_mean`
- `bal_acc_penalized`

### `stats_score`
Score sintètic de separació estadística.

Combina:
- `median_abs_rho`
- `n_sig_adj`
- `median_abs_rb`
- `median_H`
- `median_v`

### `temp_score`
Score sintètic d’estructura tèrmica.

Combina:
- `tempbin_score`
- `mean_abs_prop_diff`
- invers del solapament (`mean_ci_overlap_norm`)
- dominància per bin
- invers de l’entropia per bin

### `balance_score`
Score sintètic de qualitat del balanceig.

Combina:
- penalització de la classe més gran
- recompensa de la classe més petita
- entropia global de classes

---

## 12. Scores finals

### `empirical_score`
Score global d’**empirical robustness**.

Combina:
- `ml_score`
- `stats_score`
- `balance_score`

Interpretació:
- més alt = millor compromís entre predicció, separació estadística i robustesa de classes

### `global_score`
Score final més ampli.

Combina:
- `empirical_score`
- `temp_score`

Interpretació:
- més alt = agrupació forta tant com a estat reduït global com des del punt de vista de règim tèrmic

---

## 13. Coverage d’evidència

### `evidence_coverage`
Nombre de blocs d’evidència bàsics disponibles per a aquella fila.

Al notebook es calcula comptant quants dels següents 12 indicadors **no són NaN**:

1. `bal_acc_adj_chance`
2. `f1_macro_mean`
3. `bal_acc_penalized`
4. `median_abs_rho`
5. `n_sig_adj`
6. `median_abs_rb`
7. `median_H`
8. `median_v`
9. `tempbin_score`
10. `largest_class_share`
11. `smallest_class_share`
12. `entropy_norm`

Interpretació:
- més alt = l’outcome està recolzat per més peces d’evidència
- més baix = hi ha algun bloc que no es va poder calcular o no apareix als CSVs

Exemple:
- `comfort4_option1` té `evidence_coverage = 10` perquè li falten `n_sig_adj` i `median_abs_rb`, és a dir, el bloc de separació adjacent

In [ ]:
# ----------------------------
# ML summary
# ----------------------------
ml = ml_raw.copy()
ml["outcome"] = ml["partition"].map(partition_to_outcome)
ml = ml.dropna(subset=["outcome"]).copy()

# Keep the strongest ML row for each named outcome.
# Priority:
# 1) adjusted balanced accuracy
# 2) macro-F1
# 3) penalized balanced accuracy
ml_best = (
    ml.sort_values(
        ["bal_acc_adj_chance", "f1_macro_mean", "bal_acc_penalized"],
        ascending=[False, False, False]
    )
    .groupby("outcome", as_index=False)
    .first()
)

ml_best = ml_best[
    [
        "outcome", "feature_set", "model",
        "bal_acc_mean", "f1_macro_mean",
        "bal_acc_adj_chance", "bal_acc_penalized",
        "largest_class_share", "smallest_class_share", "entropy_norm"
    ]
].copy()

# ----------------------------
# Statistical summaries
# ----------------------------
sp_summary = (
    sp.groupby("outcome", as_index=False)
    .agg(
        n_sig_rho=("p_value", lambda s: int((s < 0.05).sum())),
        median_abs_rho=("spearman_rho", lambda s: float(np.median(np.abs(s)))),
        max_abs_rho=("spearman_rho", lambda s: float(np.max(np.abs(s))))
    )
)

pw_summary = (
    pw.groupby("outcome", as_index=False)
    .agg(
        n_adj_pairs=("p_holm", "size"),
        n_sig_adj=("p_holm", lambda s: int((s < 0.05).sum())),
        median_abs_rb=("effect_rank_biserial", lambda s: float(np.median(np.abs(s)))),
        max_abs_rb=("effect_rank_biserial", lambda s: float(np.max(np.abs(s))))
    )
)

kw_summary = (
    kw.groupby("outcome", as_index=False)
    .agg(
        n_sig_kw=("p_value", lambda s: int((s < 0.05).sum())),
        median_H=("H", "median"),
        max_H=("H", "max")
    )
)

chi_summary = (
    chi.groupby("outcome", as_index=False)
    .agg(
        n_sig_chi=("p_value", lambda s: int((s < 0.05).sum())),
        median_v=("cramers_v", "median"),
        max_v=("cramers_v", "max")
    )
)

# ----------------------------
# Merge everything
# ----------------------------
results = (
    ml_best
    .merge(sp_summary, on="outcome", how="outer")
    .merge(pw_summary, on="outcome", how="outer")
    .merge(kw_summary, on="outcome", how="outer")
    .merge(chi_summary, on="outcome", how="outer")
    .merge(temp, on="outcome", how="outer")
)

results["pretty_label"] = results["outcome"].map(pretty_labels)

results.sort_values("outcome").reset_index(drop=True)


## Turn raw metrics into component scores

Each score is normalized to the `[0, 1]` range among the candidate outcomes.

### Component scores
- **ML score**  
  Based on adjusted balanced accuracy, macro-F1, and penalized balanced accuracy

- **Stats score**  
  Based on median absolute rho, adjacent-pair significance, adjacent-pair effect size, median Kruskal H, and median Cramér's V

- **Temp score**  
  Based on the temperature-bin scorecard: thermal separation, CI overlap, dominance, and bin entropy

- **Balance score**  
  Based on largest class share, smallest class share, and normalized class entropy

### Combined scores
- **Empirical score**  
  `ML + classical statistics + balance`

- **Global score**  
  `Empirical score + thermal-regime sharpness`

This way, no single criterion dominates the selection.


In [ ]:
def minmax01(s, higher_is_better=True):
    s = pd.Series(s, dtype=float)
    out = pd.Series(np.nan, index=s.index)
    mask = s.notna()

    if mask.sum() == 0:
        return out

    ss = s[mask]
    mn, mx = ss.min(), ss.max()

    if np.isclose(mx, mn):
        out.loc[mask] = 0.5
    else:
        out.loc[mask] = (ss - mn) / (mx - mn)

    if not higher_is_better:
        out.loc[mask] = 1 - out.loc[mask]

    return out


def weighted_mean_available(score_df, cols, weights):
    """Weighted mean that ignores missing components row by row."""
    vals = pd.concat([(score_df[c] * w) for c, w in zip(cols, weights)], axis=1)
    num = vals.sum(axis=1, skipna=True)

    den = pd.concat(
        [(score_df[c].notna().astype(float) * w) for c, w in zip(cols, weights)],
        axis=1
    ).sum(axis=1)

    out = num / den
    out[den == 0] = np.nan
    return out


# ----------------------------
# ML recoverability
# ----------------------------
ml_components = pd.DataFrame({
    "bal_acc_adj_score": minmax01(results["bal_acc_adj_chance"], True),
    "f1_macro_score": minmax01(results["f1_macro_mean"], True),
    "bal_acc_penalized_score": minmax01(results["bal_acc_penalized"], True),
})

results["ml_score"] = weighted_mean_available(
    ml_components,
    cols=["bal_acc_adj_score", "f1_macro_score", "bal_acc_penalized_score"],
    weights=[0.40, 0.30, 0.30]
)

# ----------------------------
# Classical separability
# ----------------------------
stats_components = pd.DataFrame({
    "rho_score": minmax01(results["median_abs_rho"], True),
    "adj_sig_score": minmax01(results["n_sig_adj"], True),
    "adj_effect_score": minmax01(results["median_abs_rb"], True),
    "kruskal_score": minmax01(results["median_H"], True),
    "cramers_v_score": minmax01(results["median_v"], True),
})

results["stats_score"] = weighted_mean_available(
    stats_components,
    cols=["rho_score", "adj_sig_score", "adj_effect_score", "kruskal_score", "cramers_v_score"],
    weights=[0.20, 0.20, 0.20, 0.20, 0.20]
)

# ----------------------------
# Thermal-regime sharpness
# ----------------------------
temp_components = pd.DataFrame({
    "tempbin_score_norm": minmax01(results["tempbin_score"], True),
    "abs_diff_score": minmax01(results["mean_abs_prop_diff"], True),
    "overlap_score": minmax01(results["mean_ci_overlap_norm"], False),
    "dominance_score": minmax01(results["mean_dominant_prop"], True),
    "bin_entropy_score": minmax01(results["mean_entropy_norm"], False),
})

results["temp_score"] = weighted_mean_available(
    temp_components,
    cols=["tempbin_score_norm", "abs_diff_score", "overlap_score", "dominance_score", "bin_entropy_score"],
    weights=[0.30, 0.20, 0.20, 0.15, 0.15]
)

# ----------------------------
# Balance / practical robustness
# ----------------------------
balance_components = pd.DataFrame({
    "largest_penalty": minmax01(results["largest_class_share"], False),
    "smallest_bonus": minmax01(results["smallest_class_share"], True),
    "entropy_bonus": minmax01(results["entropy_norm"], True),
})

results["balance_score"] = weighted_mean_available(
    balance_components,
    cols=["largest_penalty", "smallest_bonus", "entropy_bonus"],
    weights=[0.40, 0.25, 0.35]
)

# ----------------------------
# Evidence coverage
# ----------------------------
results["evidence_coverage"] = results[
    [
        "bal_acc_adj_chance", "f1_macro_mean", "bal_acc_penalized",
        "median_abs_rho", "n_sig_adj", "median_abs_rb",
        "median_H", "median_v",
        "tempbin_score",
        "largest_class_share", "smallest_class_share", "entropy_norm"
    ]
].notna().sum(axis=1)

# ----------------------------
# Combined scores
# ----------------------------
# results["empirical_score"] = weighted_mean_available(
#     results[["ml_score", "stats_score", "balance_score"]],
#     cols=["ml_score", "stats_score", "balance_score"],
#     weights=[0.45, 0.35, 0.20]
# )

# results["global_score"] = weighted_mean_available(
#     results[["empirical_score", "temp_score"]],
#     cols=["empirical_score", "temp_score"],
#     weights=[0.65, 0.35]
# )

# results["global_score"] = weighted_mean_available(
#     results[["empirical_score", "temp_score"]],
#     cols=["empirical_score", "temp_score"],
#     weights=[0.65, 0.35]
# )
results["global_score_penalized"] = (
    0.25 * results["ml_score"] +
    0.25 * results["stats_score"] +
    0.25 * results["balance_score"]
)

# Penalització per particions massa desequilibrades
# results["global_score_penalized"] = results["global_score"] * results["balance_score"]

results = results.sort_values("global_score_penalized", ascending=False).reset_index(drop=True)

results[
    [
        "pretty_label", "ml_score", "stats_score", "temp_score", "balance_score",
       "global_score_penalized", "evidence_coverage"
    ]
].round(3)


## Main comparison table

This is the compact table I would use in the main text.  
It keeps only the most relevant metrics needed to justify the chosen coarse-grainings.


In [ ]:
main_table = (
    results[results["outcome"].isin(main_outcomes)]
    [
        [
            "pretty_label", "feature_set", "model",
            "bal_acc_mean", "f1_macro_mean", "bal_acc_adj_chance", "bal_acc_penalized",
            "largest_class_share", "smallest_class_share", "entropy_norm",
            "median_abs_rho", "n_sig_adj", "median_abs_rb", "median_H", "median_v",
            "tempbin_score", "mean_ci_overlap_norm", "mean_dominant_prop", "mean_entropy_norm",
            "ml_score", "stats_score", "temp_score", "balance_score", "empirical_score", "global_score",
            "evidence_coverage"
        ]
    ]
    .copy()
)

main_table = main_table.rename(columns={
    "pretty_label": "Outcome",
    "feature_set": "Best feature set",
    "model": "Best model",
    "bal_acc_mean": "BalAcc",
    "f1_macro_mean": "MacroF1",
    "bal_acc_adj_chance": "BalAcc_adj",
    "bal_acc_penalized": "BalAcc_pen",
    "largest_class_share": "LargestShare",
    "smallest_class_share": "SmallestShare",
    "entropy_norm": "ClassEntropy",
    "median_abs_rho": "Median|rho|",
    "n_sig_adj": "AdjSig",
    "median_abs_rb": "Median|RB|",
    "median_H": "MedianH",
    "median_v": "MedianV",
    "tempbin_score": "TempBinScore",
    "mean_ci_overlap_norm": "MeanNormOverlap",
    "mean_dominant_prop": "MeanDominantProp",
    "mean_entropy_norm": "MeanBinEntropy",
    "ml_score": "MLscore",
    "stats_score": "StatsScore",
    "temp_score": "TempScore",
    "balance_score": "BalanceScore",
    "empirical_score": "EmpiricalScore",
    "global_score": "GlobalScore",
    "evidence_coverage": "Coverage",
})

main_table = main_table.round(3)

main_table.to_csv(CSV_DIR / "results_main_comparison_table.csv", index=False)

main_table


## Full appendix table

This keeps the richer set of metrics for the appendix or for later discussion while writing the thesis text.


In [ ]:
appendix_table = results[
    [
        "pretty_label", "outcome", "feature_set", "model",
        "bal_acc_mean", "f1_macro_mean", "bal_acc_adj_chance", "bal_acc_penalized",
        "largest_class_share", "smallest_class_share", "entropy_norm",
        "n_sig_rho", "median_abs_rho", "max_abs_rho",
        "n_adj_pairs", "n_sig_adj", "median_abs_rb", "max_abs_rb",
        "n_sig_kw", "median_H", "max_H",
        "n_sig_chi", "median_v", "max_v",
        "mean_abs_prop_diff", "mean_ci_overlap", "mean_ci_overlap_norm", "max_abs_prop_diff", "min_ci_overlap",
        "mean_dominant_prop", "max_dominant_prop", "mean_entropy_norm", "min_entropy_norm",
        "mean_concentration", "mean_l1_imbalance", "tempbin_score",
        "ml_score", "stats_score", "temp_score", "balance_score", "empirical_score", "global_score",
        "evidence_coverage"
    ]
].round(3)

appendix_table.to_csv(CSV_DIR / "results_appendix_full_table.csv", index=False)

appendix_table


## Ranking table

This is the shortest table to use when you want one line per candidate outcome with only the component scores.


In [ ]:
rank_table = (
    results[results["outcome"].isin(main_outcomes)]
    [["pretty_label", "ml_score", "stats_score", "temp_score", "balance_score", "empirical_score", "global_score", "evidence_coverage"]]
    .sort_values("global_score", ascending=False
    )
    .round(3)
)

rank_table.to_csv(CSV_DIR / "candidate_groupings_rank_table.csv", index=False)

rank_table


## Plot 1 — component heatmap

This is the most compact visualization of the whole comparison.

Interpretation:
- **ML** = recoverability from grouped machine learning
- **Stats** = classical inferential separation
- **Temp-bin** = thermal-regime sharpness
- **Balance** = practical robustness of the grouped state distribution
- **Global** = overall synthesis score


In [ ]:
plot_order = [
    "comfort3",
    "comfort3_option1",
    "comfort3_option2",
    "comfort4_soft",
    "comfort4_option1",
    "comfort3_UNoption",
    "comfort4_option2",
]

heat_df = results.set_index("outcome").loc[
    [o for o in plot_order if o in results["outcome"].values]
].reset_index()

score_cols = ["ml_score", "stats_score", "temp_score", "balance_score", "global_score"]
score_labels = ["ML", "Stats", "Temp-bin", "Balance", "Global"]

M = heat_df[score_cols].to_numpy()

fig, ax = plt.subplots(figsize=(7.4, max(4.2, len(heat_df) * 0.62)))
im = ax.imshow(M, aspect="auto")

ax.set_xticks(range(len(score_cols)))
ax.set_xticklabels(score_labels)
ax.set_yticks(range(len(heat_df)))
ax.set_yticklabels([pretty_labels.get(o, o) for o in heat_df["outcome"]])

for i in range(M.shape[0]):
    for j in range(M.shape[1]):
        v = M[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=8)

ax.set_title("Candidate coarse-grainings: component scores")
cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
cbar.set_label("Normalized score")

plt.tight_layout()
plt.savefig(FIG_DIR / "candidate_groupings_component_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()


## Plot 2 — tradeoff diagram

This plot is designed to look like a simple phase-diagram-style tradeoff:

- **x-axis** = empirical robustness  
  (ML + classical statistics + balance)

- **y-axis** = thermal-regime sharpness  
  (temperature-bin evidence)

- **bubble size** = balance score

This is useful because it immediately shows the tension you have been discussing:
some groupings are more **globally robust**, while others are more **thermally sharp**.


In [ ]:
trade_df = results[results["outcome"].isin(main_outcomes)].copy()

fig, ax = plt.subplots(figsize=(7.0, 5.5))

for _, row in trade_df.iterrows():
    x = row["empirical_score"]
    y = row["temp_score"]
    size = 350 + 650 * row["balance_score"]

    ax.scatter(x, y, s=size, alpha=0.82)
    ax.text(
        x + 0.012,
        y + 0.006,
        row["pretty_label"],
        fontsize=9
    )

ax.set_xlabel("Empirical robustness score")
ax.set_ylabel("Thermal-regime sharpness score")
ax.set_title("Tradeoff between robustness and regime structure")
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(-0.02, 1.02)
ax.grid(True, alpha=0.30)

ax.axvline(trade_df["empirical_score"].median(), linestyle="--", alpha=0.35)
ax.axhline(trade_df["temp_score"].median(), linestyle="--", alpha=0.35)

plt.tight_layout()
plt.savefig(FIG_DIR / "candidate_groupings_tradeoff_phase_diagram.png", dpi=220, bbox_inches="tight")
plt.show()


## Optional plot 3 — scorecard bars

A simple bar-chart summary can also be useful for the thesis text or slides.


In [ ]:
scorecard_plot = (
    results[results["outcome"].isin(main_outcomes)]
    [["pretty_label", "global_score", "empirical_score", "temp_score"]]
    .sort_values("global_score", ascending=True)
    .reset_index(drop=True)
)

fig, ax = plt.subplots(figsize=(7.4, 4.8))
y = np.arange(len(scorecard_plot))

ax.barh(y, scorecard_plot["global_score"], alpha=0.80, label="Global")
ax.scatter(scorecard_plot["empirical_score"], y, s=70, label="Empirical")
ax.scatter(scorecard_plot["temp_score"], y, s=70, label="Temp-bin")

ax.set_yticks(y)
ax.set_yticklabels(scorecard_plot["pretty_label"])
ax.set_xlabel("Normalized score")
ax.set_title("Summary scorecard for the main candidate outcomes")
ax.grid(axis="x", alpha=0.25)
ax.legend()

plt.tight_layout()
plt.savefig(FIG_DIR / "candidate_groupings_summary_scorecard.png", dpi=220, bbox_inches="tight")
plt.show()


## Suggested interpretation for the thesis text

A clean way to describe this section is:

> We treat each grouped outcome as a candidate coarse-graining of the original seven-level comfort state space.  
> To compare them, we evaluate four complementary properties: recoverability from grouped machine learning, classical inferential separability, thermal-regime sharpness across temperature bins, and class-balance robustness.  
> This avoids selecting a grouping on the basis of a single criterion such as raw accuracy or class balance alone.

Then, once you inspect the tables and figures, you can separate two different notions of “best” if needed:

1. **Best global empirical grouping**  
2. **Best thermally sharp / physics-ready grouping**

That is a much stronger bridge into the later confusion-matrix and dynamics sections.


## Next step after this notebook

After inspecting the tables and plots here, the next clean step is:

1. build confusion matrices for the shortlisted outcomes
2. decide the main 3-group and 4-group state spaces
3. move into transitions / Markov structure / phase-like dynamics using the most physics-ready coarse-graining
